In [1]:
from google.colab import files # Import files

uploaded = files.upload() # Upload files Arizona Diamondbacks Batters
uploaded = files.upload() #

Saving A - MLB STATS.csv to A - MLB STATS.csv


Saving mlb_team_pitching_statistics.csv to mlb_team_pitching_statistics.csv


In [2]:
import pandas as pd # Import pandas library
import numpy as np # Import numpy library
import tensorflow as tf # Import tensorflow library

try:
    # Added skiprows=1 to account for the extra header line in the CSV
    df_database = pd.read_csv('A - MLB STATS.csv', skiprows=1) # Read the csv file into a DataFrame
    print("Dataset loaded successfully!") # Print success message
except FileNotFoundError: # Catches the specific error that occurs if the specified file is not found
    print("Error: Converted csv file not found. Please ensure it's uploaded and the filename is correct.") # Print error if file is not found
except Exception as e: # Catches any other general exceptions that might occur during file reading.
    print(f"An error occurred while reading the file: {e}")

# Display the DataFrame if it was loaded successfully
if 'df_database' in locals() and not df_database.empty: # Checks if 'df_database' was successfully created and is not empty
    display(df_database) # Display the first few rows of the DataFrame
else:
    print("DataFrame could not be loaded or is empty.")

Dataset loaded successfully!


,Tm,#Bat,BatAge,R/G,G,PA,AB,R,H,2B,...,SLG,OPS,OPS+,TB,GDP,HBP,SH,SF,IBB,LOB
0,Arizona Diamondbacks,65,27.8,4.88,162,6210,5480,791,1377,277,...,0.433,0.757,108,2372,109,81,37,64,21,1112
1,Athletics,58,26.1,4.52,162,6151,5547,733,1403,296,...,0.431,0.749,102,2388,117,44,18,35,14,1100
2,Atlanta Braves,71,28.3,4.47,162,6186,5508,724,1349,243,...,0.399,0.720,101,2200,98,51,15,36,14,1160
3,Baltimore Orioles,70,26.5,4.18,162,6020,5416,677,1273,251,...,0.394,0.699,95,2135,102,75,4,41,14,1044
4,Boston Red Sox,56,27.6,4.85,162,6206,5562,786,1414,324,...,0.421,0.745,109,2344,99,72,13,41,24,1125
5,Chicago Cubs,52,28.4,4.90,162,6162,5495,793,1371,267,...,0.430,0.751,117,2365,72,46,10,55,16,1108
6,Chicago White Sox,62,26.8,3.99,162,5987,5377,647,1250,243,...,0.373,0.675,88,2008,108,49,22,36,8,1055
7,Cincinnati Reds,53,27.2,4.42,162,6082,5448,716,1333,250,...,0.391,0.706,91,2130,106,51,12,44,15,1088
8,Cleveland Guardians,45,27.4,3.97,162,5943,5310,643,1199,243,...,0.373,0.670,85,1982,78,60,28,50,30,1016
9,Colorado Rockies,57,26.7,3.69,162,5908,5408,597,1281,253,...,0.386,0.679,80,2088,118,48,21,35,5,986


In [3]:
print(df_database.columns.tolist())

['Tm', '#Bat', 'BatAge', 'R/G', 'G', 'PA', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'BA', 'OBP', 'SLG', 'OPS', 'OPS+', 'TB', 'GDP', 'HBP', 'SH', 'SF', 'IBB', 'LOB']


In [4]:
# Drop league average and totals rows
df_database = df_database[~df_database['Tm'].isin(['League Average', 'League Totals'])]

# Reset index
df_database = df_database.reset_index(drop=True)

# Convert all stat columns to numeric just in case
stat_cols = ['#Bat', 'BatAge', 'R/G', 'G', 'PA', 'AB', 'R', 'H', '2B', '3B', 'HR',
             'RBI', 'SB', 'CS', 'BB', 'SO', 'BA', 'OBP', 'SLG', 'OPS', 'OPS+',
             'TB', 'GDP', 'HBP', 'SH', 'SF', 'IBB', 'LOB']

df_database[stat_cols] = df_database[stat_cols].apply(pd.to_numeric, errors='coerce')
# Applies the `pd.to_numeric` function to all columns specified in `stat_cols`. `errors='coerce'` will convert any non-numeric values to `NaN` (Not a Number) instead of raising an error

# Check for missing values
print("Missing values per column:")
print(df_database.isnull().sum()) # Calculates and prints the count of missing (NaN) values for each column in the `df_database` DataFrame
print(f"\nDataset shape: {df_database.shape}")
print(f"\nTeams loaded: {df_database['Tm'].tolist()}")


Missing values per column:
Tm        0
#Bat      0
BatAge    0
R/G       0
G         0
PA        0
AB        0
R         0
H         0
2B        0
3B        0
HR        0
RBI       0
SB        0
CS        0
BB        0
SO        0
BA        0
OBP       0
SLG       0
OPS       0
OPS+      0
TB        0
GDP       0
HBP       0
SH        0
SF        0
IBB       0
LOB       0
dtype: int64

Dataset shape: (30, 29)

Teams loaded: ['Arizona Diamondbacks', 'Athletics', 'Atlanta Braves', 'Baltimore Orioles', 'Boston Red Sox', 'Chicago Cubs', 'Chicago White Sox', 'Cincinnati Reds', 'Cleveland Guardians', 'Colorado Rockies', 'Detroit Tigers', 'Houston Astros', 'Kansas City Royals', 'Los Angeles Angels', 'Los Angeles Dodgers', 'Miami Marlins', 'Milwaukee Brewers', 'Minnesota Twins', 'New York Mets', 'New York Yankees', 'Philadelphia Phillies', 'Pittsburgh Pirates', 'San Diego Padres', 'Seattle Mariners', 'San Francisco Giants', 'St. Louis Cardinals', 'Tampa Bay Rays', 'Texas Rangers', 'Toronto Blu

In [5]:
from sklearn.preprocessing import MinMaxScaler # Imports the MinMaxScaler class from scikit-learn's preprocessing module. This is used to scale numerical features to a specified range, typically 0 to 1

# Select the most meaningful features for predicting offensive matchup strength
features = ['R/G', 'OPS', 'SLG', 'OPS+', 'HR', 'BB', 'SO', 'SB', 'BA']

# Scale the features between 0 and 1
scaler = MinMaxScaler()
df_scaled = df_database.copy()
df_scaled = df_database.copy()
df_scaled[features] = scaler.fit_transform(df_database[features])

# Build a team strength score from the scaled features
weights = {
    'R/G':  0.40,
    'OPS':  0.10,
    'SLG':  0.10,
    'OPS+': 0.10,
    'OBP':  0.05,
    'HR':   0.10,
    'BB':   0.05,
    'SO':  -0.05,
    'SB':   0.10,
    'BA':   0.05
}
# Calculates a 'strength_score' for each team
df_scaled['strength_score'] = sum(df_scaled[col] * w for col, w in weights.items())

# Predictor function
def predict_matchup(team1, team2):
    teams = df_scaled['Tm'].tolist()

    # Fuzzy match in case of slight name differences
    t1 = [t for t in teams if team1.lower() in t.lower()]
    t2 = [t for t in teams if team2.lower() in t.lower()]

    if not t1:
        print(f"Could not find team: {team1}")
        return
    if not t2:
        print(f"Could not find team: {team2}")
        return

    t1_name = t1[0]
    t2_name = t2[0]

    s1 = df_scaled[df_scaled['Tm'] == t1_name]['strength_score'].values[0]
    s2 = df_scaled[df_scaled['Tm'] == t2_name]['strength_score'].values[0]

    total = s1 + s2
    prob1 = round((s1 / total) * 100, 1)
    prob2 = round((s2 / total) * 100, 1)

    print(f"  MATCHUP PREDICTION")

    print(f"  {t1_name}: {prob1}% win probability")

    print(f"  {t2_name}: {prob2}% win probability")

    if prob1 > prob2:
        print(f"  Predicted Winner: {t1_name}")
    elif prob2 > prob1:
        print(f"  Predicted Winner: {t2_name}")
    else:
        print("  Too close to call")

predict_matchup("New York Yankees", "Los Angeles Dodgers")


  MATCHUP PREDICTION
  New York Yankees: 53.6% win probability
  Los Angeles Dodgers: 46.4% win probability
  Predicted Winner: New York Yankees


In [6]:
import pandas as pd # Import pandas library
import numpy as np # Import numpy library
import tensorflow as tf # Import tensorflow library

try:
    df_database = pd.read_csv('mlb_team_pitching_statistics.csv') # Read the csv file into a DataFrame
    print("Dataset loaded successfully!") # Print success message
except FileNotFoundError:
    print("Error: Converted csv file not found. Please ensure it's uploaded and the filename is correct.") # Print error if file is not found
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

# Display the DataFrame if it was loaded successfully
if 'df_database' in locals() and not df_database.empty:
    display(df_database) # Display the first few rows of the DataFrame
else:
    print("DataFrame could not be loaded or is empty.")

Dataset loaded successfully!


,Tm,#P,PAge,RA/G,W,L,W-L%,ERA,G,GS,...,BF,ERA+,FIP,WHIP,H9,HR9,BB9,SO9,SO/W,LOB
0,Arizona Diamondbacks,43,29.8,4.85,80,82,0.494,4.49,162,162,...,6183,96,4.27,1.318,8.7,1.2,3.1,8.0,2.58,1093
1,Athletics,36,28.6,5.04,76,86,0.469,4.70,162,162,...,6232,95,4.66,1.361,8.7,1.4,3.6,8.3,2.32,1120
2,Atlanta Braves,46,29.4,4.53,76,86,0.469,4.36,162,162,...,6116,94,4.20,1.306,8.4,1.2,3.3,8.9,2.67,1088
3,Baltimore Orioles,42,30.6,4.86,75,87,0.463,4.60,162,162,...,6166,88,4.41,1.365,9.0,1.4,3.3,8.5,2.58,1098
4,Boston Red Sox,36,29.4,4.17,89,73,0.549,3.70,162,162,...,6150,109,3.98,1.286,8.3,1.0,3.3,8.5,2.57,1156
5,Chicago Cubs,34,31.0,4.01,92,70,0.568,3.79,162,162,...,5904,102,4.16,1.177,8.1,1.3,2.5,7.9,3.12,965
6,Chicago White Sox,33,27.5,4.58,60,102,0.370,4.26,162,162,...,6124,99,4.47,1.364,8.5,1.2,3.8,8.2,2.16,1148
7,Cincinnati Reds,29,29.1,4.20,83,79,0.512,3.86,162,162,...,6035,118,4.11,1.222,7.9,1.2,3.1,8.7,2.79,1070
8,Cleveland Guardians,26,27.1,4.01,88,74,0.543,3.70,162,162,...,6054,111,3.95,1.261,8.1,1.1,3.3,8.6,2.62,1098
9,Colorado Rockies,31,27.7,6.30,43,119,0.265,5.97,162,162,...,6367,81,5.24,1.582,10.7,1.6,3.5,7.0,1.97,1135


In [7]:
print(df_database.columns.tolist())

['Tm', '#P', 'PAge', 'RA/G', 'W', 'L', 'W-L%', 'ERA', 'G', 'GS', 'GF', 'CG', 'tSho', 'cSho', 'SV', 'IP', 'H', 'R', 'ER', 'HR', 'BB', 'IBB', 'SO', 'HBP', 'BK', 'WP', 'BF', 'ERA+', 'FIP', 'WHIP', 'H9', 'HR9', 'BB9', 'SO9', 'SO/W', 'LOB']


In [8]:
# Drop league average and totals rows
df_database = df_database[~df_database['Tm'].isin(['League Average', 'League Totals'])]

# Reset index
df_database = df_database.reset_index(drop=True)

# Convert all stat columns to numeric just in case
stat_cols = ['#P', 'PAge', 'RA/G', 'W', 'L', 'W-L%', 'ERA', 'G', 'GS', 'GF', 'CG', 'tSho', 'cSho', 'SV', 'IP', 'H', 'R', 'ER', 'HR', 'BB', 'IBB', 'SO', 'HBP', 'BK', 'WP', 'BF', 'ERA+', 'FIP', 'WHIP', 'H9', 'HR9', 'BB9', 'SO9', 'SO/W', 'LOB']

# drops rows from `df_database` where the 'Tm' (Team) column has missing (NaN) values
df_database = df_database.dropna(subset=['Tm'])

df_database[stat_cols] = df_database[stat_cols].apply(pd.to_numeric, errors='coerce')

# Check for missing values
print("Missing values per column:")
print(df_database.isnull().sum())
print(f"\nDataset shape: {df_database.shape}")
print(f"\nTeams loaded: {df_database['Tm'].tolist()}")


Missing values per column:
Tm      0
#P      0
PAge    0
RA/G    0
W       0
L       0
W-L%    0
ERA     0
G       0
GS      0
GF      0
CG      0
tSho    0
cSho    0
SV      0
IP      0
H       0
R       0
ER      0
HR      0
BB      0
IBB     0
SO      0
HBP     0
BK      0
WP      0
BF      0
ERA+    0
FIP     0
WHIP    0
H9      0
HR9     0
BB9     0
SO9     0
SO/W    0
LOB     0
dtype: int64

Dataset shape: (30, 36)

Teams loaded: ['Arizona Diamondbacks', 'Athletics', 'Atlanta Braves', 'Baltimore Orioles', 'Boston Red Sox', 'Chicago Cubs', 'Chicago White Sox', 'Cincinnati Reds', 'Cleveland Guardians', 'Colorado Rockies', 'Detroit Tigers', 'Houston Astros', 'Kansas City Royals', 'Los Angeles Angels', 'Los Angeles Dodgers', 'Miami Marlins', 'Milwaukee Brewers', 'Minnesota Twins', 'New York Mets', 'New York Yankees', 'Philadelphia Phillies', 'Pittsburgh Pirates', 'San Diego Padres', 'Seattle Mariners', 'San Francisco Giants', 'St. Louis Cardinals', 'Tampa Bay Rays', 'Texas Rangers', 

In [9]:
from sklearn.preprocessing import MinMaxScaler

# Select the most meaningful features for predicting offensive matchup strength
features = ['RA/G', 'ERA', 'tSho', 'cSho', 'ERA+', 'FIP', 'WHIP', 'H9', 'HR9', 'BB9', 'SO9']

# Scale the features between 0 and 1
scaler = MinMaxScaler()
df_scaled = df_database.copy()
df_scaled = df_database.copy()
df_scaled[features] = scaler.fit_transform(df_database[features])

# Build a team strength score from the scaled features
weights = {
    'RA/G':  0.50,
    'ERA':  0.20,
    'tSho':  0.05,
    'cSho': 0.05,
    'ERA+':  0.10,
    'FIP':   0.05,
    'WHIP':   0.10,
    'H9':  -0.05,
    'HR9':   -0.05,
    'BB9':   -0.05,
    'SO9':  0.10
}

# Calculates a 'strength_score' for each team
df_scaled['strength_score'] = sum(df_scaled[col] * w for col, w in weights.items())

# Predictor function
def predict_matchup(team1, team2):
    teams = df_scaled['Tm'].tolist()

    # Fuzzy match in case of slight name differences --> so that you don't have to type in the name exactly as it is in the data file
    t1 = [t for t in teams if team1.lower() in t.lower()]
    t2 = [t for t in teams if team2.lower() in t.lower()]

    if not t1:
        print(f"Could not find team: {team1}")
        return
    if not t2:
        print(f"Could not find team: {team2}")
        return

    t1_name = t1[0]
    t2_name = t2[0]

    s1 = df_scaled[df_scaled['Tm'] == t1_name]['strength_score'].values[0]
    s2 = df_scaled[df_scaled['Tm'] == t2_name]['strength_score'].values[0]

    total = s1 + s2
    prob1 = round((s1 / total) * 100, 1)
    prob2 = round((s2 / total) * 100, 1)

    print(f"  MATCHUP PREDICTION")

    print(f"  {t1_name}: {prob1}% win probability")

    print(f"  {t2_name}: {prob2}% win probability")

    if prob1 > prob2:
        print(f"  Predicted Winner: {t1_name}")
    elif prob2 > prob1:
        print(f"  Predicted Winner: {t2_name}")
    else:
        print("  Too close to call")

predict_matchup("Colorado Rockies", "Los Angeles Dodgers")


  MATCHUP PREDICTION
  Colorado Rockies: 71.0% win probability
  Los Angeles Dodgers: 29.0% win probability
  Predicted Winner: Colorado Rockies


In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Load batting data
batting_df = pd.read_csv('A - MLB STATS.csv', skiprows=1)

# Remove unnecessary rows
batting_df = batting_df[
    ~batting_df['Tm'].isin(['League Average', 'League Totals'])
]

# Reset index
batting_df = batting_df.reset_index(drop=True)

# Ensure team names stay strings
batting_df['Tm'] = batting_df['Tm'].astype(str)

# Offensive stat columns
batting_stat_cols = [
    '#Bat', 'BatAge', 'R/G', 'G', 'PA', 'AB', 'R', 'H',
    '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO',
    'BA', 'OBP', 'SLG', 'OPS', 'OPS+', 'TB', 'GDP',
    'HBP', 'SH', 'SF', 'IBB', 'LOB'
]

# Convert stats to numeric
batting_df[batting_stat_cols] = batting_df[batting_stat_cols].apply(
    pd.to_numeric,
  # converts unvalid data into a missing value
    errors='coerce'
)

# Offensive features
offensive_features = [
    'R/G', 'OPS', 'SLG', 'OPS+', 'HR',
    'BB', 'SO', 'SB', 'BA'
]

# Scale offensive features
offensive_scaler = MinMaxScaler()

batting_scaled = batting_df.copy()

batting_scaled[offensive_features] = offensive_scaler.fit_transform(
    batting_df[offensive_features]
)

# Offensive weights
offensive_weights = {
    'R/G':  0.40,
    'OPS':  0.10,
    'SLG':  0.10,
    'OPS+': 0.10,
    'HR':   0.10,
    'BB':   0.05,
    'SO':  -0.05,
    'SB':   0.05,
    'BA':   0.15
}

# Calculate offensive score
batting_scaled['offense_score'] = sum(
    batting_scaled[col] * w
    for col, w in offensive_weights.items()
)

# Load pitching data
pitching_df = pd.read_csv('mlb_team_pitching_statistics.csv')

# Remove unnecessary rows
pitching_df = pitching_df[
    ~pitching_df['Tm'].isin(['League Average', 'League Totals'])
]

# Reset index
pitching_df = pitching_df.reset_index(drop=True)

# Keep team names as strings
pitching_df['Tm'] = pitching_df['Tm'].astype(str)

# Pitching stat columns
pitching_stat_cols = [
    '#P', 'PAge', 'RA/G', 'W', 'L', 'W-L%',
    'ERA', 'G', 'GS', 'GF', 'CG', 'tSho',
    'cSho', 'SV', 'IP', 'H', 'R', 'ER',
    'HR', 'BB', 'IBB', 'SO', 'HBP', 'BK',
    'WP', 'BF', 'ERA+', 'FIP', 'WHIP',
    'H9', 'HR9', 'BB9', 'SO9', 'SO/W', 'LOB'
]

# Convert pitching stats to numeric
pitching_df[pitching_stat_cols] = pitching_df[pitching_stat_cols].apply(
    pd.to_numeric,
    errors='coerce'
)

# Pitching features
pitching_features = [
    'RA/G', 'ERA', 'ERA+', 'FIP',
    'WHIP', 'H9', 'HR9', 'BB9', 'SO9'
]

# Scale pitching features
pitching_scaler = MinMaxScaler()

pitching_scaled = pitching_df.copy()

pitching_scaled[pitching_features] = pitching_scaler.fit_transform(
    pitching_df[pitching_features]
)

# Invert stats where LOWER is better
inverse_stats = ['RA/G', 'ERA', 'FIP', 'WHIP', 'H9', 'HR9', 'BB9']

for stat in inverse_stats:
    pitching_scaled[stat] = 1 - pitching_scaled[stat]

# Pitching weights
pitching_weights = {
    'RA/G':  0.25,
    'ERA':   0.25,
    'ERA+':  0.15,
    'FIP':   0.10,
    'WHIP':  0.10,
    'H9':    0.05,
    'HR9':   0.03,
    'BB9':   0.02,
    'SO9':   0.05
}

# Calculate pitching score
pitching_scaled['pitching_score'] = sum(
    pitching_scaled[col] * w
    for col, w in pitching_weights.items()
)

# Clean team names
batting_scaled['Tm'] = batting_scaled['Tm'].str.strip()
pitching_scaled['Tm'] = pitching_scaled['Tm'].str.strip()

# Merge datasets
combined_df = pd.merge(
    batting_scaled[['Tm', 'offense_score']],
    pitching_scaled[['Tm', 'pitching_score']],
    on='Tm',
    how='inner'
)

# Final combined score
combined_df['total_score'] = (
    combined_df['offense_score'] * 0.55 +
    combined_df['pitching_score'] * 0.45
)

# Matchup predictor
def predict_matchup(team1, team2):

    teams = combined_df['Tm'].tolist()

    # "Fuzzy matching"
    t1 = [t for t in teams if team1.lower() in t.lower()]
    t2 = [t for t in teams if team2.lower() in t.lower()]

    if not t1:
        print(f"Could not find team: {team1}")
        return

    if not t2:
        print(f"Could not find team: {team2}")
        return

    t1_name = t1[0]
    t2_name = t2[0]

    s1 = combined_df[
        combined_df['Tm'] == t1_name
    ]['total_score'].values[0]

    s2 = combined_df[
        combined_df['Tm'] == t2_name
    ]['total_score'].values[0]

    total = s1 + s2

    prob1 = round((s1 / total) * 100, 1)
    prob2 = round((s2 / total) * 100, 1)

    print("\nMLB MATCHUP PREDICTION\n")

    print(f"{t1_name}: {prob1}% win probability")
    print(f"{t2_name}: {prob2}% win probability\n")

    if prob1 > prob2:
        print(f"Predicted Winner: {t1_name}")

    elif prob2 > prob1:
        print(f"Predicted Winner: {t2_name}")

    else:
        print("Too close to call")

# Prompt the user for team names
print("Available teams:")
print(", ".join(combined_df["Tm"].tolist()))

team1_input = input("\nEnter the first team: ")
team2_input = input("Enter the second team: ")

predict_matchup(team1_input, team2_input)

Available teams:
Arizona Diamondbacks, Athletics, Atlanta Braves, Baltimore Orioles, Boston Red Sox, Chicago Cubs, Chicago White Sox, Cincinnati Reds, Cleveland Guardians, Colorado Rockies, Detroit Tigers, Houston Astros, Kansas City Royals, Los Angeles Angels, Los Angeles Dodgers, Miami Marlins, Milwaukee Brewers, Minnesota Twins, New York Mets, New York Yankees, Philadelphia Phillies, Pittsburgh Pirates, San Diego Padres, Seattle Mariners, San Francisco Giants, St. Louis Cardinals, Tampa Bay Rays, Texas Rangers, Toronto Blue Jays, Washington Nationals

Enter the first team: new york mets
Enter the second team: new york yankees

MLB MATCHUP PREDICTION

New York Mets: 45.2% win probability
New York Yankees: 54.8% win probability

Predicted Winner: New York Yankees


James and Grant

All Ryan